# 2015 Philippine Customs Data Analysis

## 1. Imports

In [ ]:
import sys
sys.path.append("..") 

import config
from src.data_loader import CustomsDataLoader
from src.audit import new_audit_record, missing_values_report
from src.data_filtering import filter_and_transform_data
from src.summary import DataSummarizer
from src.benchmark import run_numpy_benchmark
from src.visualizations import generate_bar_chart, generate_heatmap
from src.validation import build_and_check_validation, export_audit_log

config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
audit_records = []

## 2. Load and validate the raw file

In [ ]:
loader = CustomsDataLoader(config.DATA_PATH, config.REQUIRED_COLUMNS)
loader.load()
loader.validate_columns()

raw_df = loader.data
audit_records.append(
    new_audit_record(1, "load_csv", "loaded raw file, no filtering yet", 0, loader.raw_row_count)
)

info = loader.summary_info()
print(f"Loaded {info['row_count']} rows, {info['column_count']} columns.")
raw_df.head()

### Missing value check on required columns

In [ ]:
nulls = missing_values_report(raw_df, columns=list(config.REQUIRED_COLUMNS))
nulls

## 3. Filter and create derived columns

In [ ]:
filtered_df, excluded_df = filter_and_transform_data(
    raw_df,
    config.CATEGORY_COL_1,
    config.MEASURE_COL,
    min_value=config.FILTER_CONFIG["min_dutiable_value"],
    excluded_countries=config.FILTER_CONFIG["excluded_countries"],
)
audit_records.append(
    new_audit_record(
        2, "filter", "measure >= min_dutiable_value, country not excluded",
        len(raw_df), len(filtered_df),
    )
)
print(f"Filtered to {len(filtered_df)} rows, excluded {len(excluded_df)}.")
filtered_df[[config.CATEGORY_COL_1, config.MEASURE_COL, "dutiablevalue_kphp", "is_high_value"]].head()

## 4. NumPy loop vs vectorized benchmark

In [ ]:
benchmark_res = run_numpy_benchmark(filtered_df[config.MEASURE_COL])
benchmark_res

## 5. Summary tables

In [ ]:
summarizer = DataSummarizer(
    filtered_df, config.CATEGORY_COL_1, config.CATEGORY_COL_2, config.MEASURE_COL
)
grouped_df, top10_df = summarizer.generate_single_grouped(config.OUTPUT_DIR)
grouped_two_df = summarizer.generate_two_grouped(config.OUTPUT_DIR)
pivot_df = summarizer.generate_pivot(config.OUTPUT_DIR)

audit_records.append(
    new_audit_record(3, "summarize", "grouped, two-factor grouped, pivot generated",
                      len(filtered_df), len(grouped_df))
)

top10_df

In [ ]:
grouped_two_df.head()

In [ ]:
pivot_df.head()

## 6. Plots
Bar chart: top 10 countries by total dutiable value. Heatmap: dutiable value by country and TQ category.

In [ ]:
generate_bar_chart(top10_df, config.CATEGORY_COL_1, config.OUTPUT_DIR / "bar.png")
generate_heatmap(pivot_df, config.OUTPUT_DIR / "heatmap.png")
print("Plots saved to output/")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(config.OUTPUT_DIR / "bar.png")))

In [ ]:
display(Image(filename=str(config.OUTPUT_DIR / "heatmap.png")))

## 7. Validation checks and audit log

In [ ]:
validation_df = build_and_check_validation(
    raw_df, filtered_df, excluded_df, grouped_df, pivot_df, benchmark_res,
    config.MEASURE_COL, config.OUTPUT_DIR,
    ref_rows=config.REFERENCE_ROW_COUNT, ref_sum=config.REFERENCE_DUTIABLE_SUM,
)
validation_df

In [ ]:
export_audit_log(audit_records, config.OUTPUT_DIR)
import pandas as pd
pd.DataFrame(audit_records)

## Summary

All outputs regenerated in `output/`: `grouped.csv`, `grouped_two.csv`, `pivot.csv`, `top10.csv`, `bar.png`, `heatmap.png`, `validation.csv`, `audit_log.csv`. 